
# 01 — Foreign Exchange API Data Ingestion

## Project: Financial Market Intelligence & Currency Risk Analytics

### Business Objective

An Indian company makes international import payments in USD, EUR, and GBP. Changes in foreign exchange rates can increase or decrease the INR cost of these payments.

The Treasury and Finance teams need reliable historical exchange-rate data to monitor currency movements and evaluate potential import-cost exposure.

### Data Source

- Provider: Frankfurter API
- Currency pairs: USD/INR, EUR/INR, GBP/INR
- Frequency: Daily available exchange rates
- Measurement: INR required to purchase one unit of foreign currency

### Technical Objectives

1. Connect to the Frankfurter API.
2. Retrieve historical exchange-rate observations.
3. Standardize the API responses into a tabular dataset.
4. Validate response status, structure, and record counts.
5. Save raw API data for reproducibility.
6. Prepare the data for further validation and PostgreSQL loading.

### Expected Output

A historical FX dataset containing:

- observation_date
- base_currency
- quote_currency
- exchange_rate
- source

### Business Interpretation

An increase in USD/INR, EUR/INR, or GBP/INR means the Indian rupee has weakened against that currency, making a fixed foreign-currency import payment more expensive in INR.

In [1]:

from pathlib import Path
import sys

# Locate the project root from the notebook working directory.
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parents[1]

elif project_root.name.lower() == "python":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Python.Scripts.pipeline_logger import get_logger

logger = get_logger("FX_API_Ingestion")

logger.info("FX API ingestion notebook initialized")


2026-11-02 15:54:01,830 | INFO | FX_API_Ingestion | FX API ingestion notebook initialized



## 1. Environment Setup and API Configuration

### Objective
Import the Python libraries required for API ingestion and load the project's environment configuration.

### Method
- Use `requests` to communicate with the Frankfurter API.
- Use `pandas` to process API responses into structured tables.
- Use `python-dotenv` to load configuration from the project's `.env` file.
- Use `pathlib` to locate project folders reliably.

### Validation
Confirm that the Frankfurter API base URL is configured before making any API requests.

### Expected Result
The environment is initialized and ready for FX data extraction.

In [1]:

from pathlib import Path
import os

import pandas as pd
import requests
from dotenv import load_dotenv

# Locate the project root from the notebook directory
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

elif not (PROJECT_ROOT / ".env").exists():
    raise FileNotFoundError(
        "Project root not found. Check your Jupyter working directory."
    )

# Load environment configuration
load_dotenv(PROJECT_ROOT / ".env")

# Retrieve Frankfurter API configuration
FRANKFURTER_BASE_URL = os.getenv("FRANKFURTER_BASE_URL")

# Validate configuration
if not FRANKFURTER_BASE_URL:
    raise ValueError("FRANKFURTER_BASE_URL is missing from .env")

print("Environment Status: READY")
print("Project Root:", PROJECT_ROOT)
print("Frankfurter API:", FRANKFURTER_BASE_URL)
print("Pandas Version:", pd.__version__)


Environment Status: READY
Project Root: E:\BA Projects\Financial_Market_Intelligence_Analytics
Frankfurter API: https://api.frankfurter.dev/v1
Pandas Version: 3.0.6



## 2. Historical USD/INR API Extraction

### Business Objective
Retrieve historical USD/INR exchange rates to understand how fluctuations in the US dollar affect an Indian company's import payment costs.

### Method
- Connect to the Frankfurter historical exchange-rate API.
- Request USD as the base currency and INR as the quote currency.
- Retrieve observations from 2026-05-21 to 2026-10-13.
- Inspect the HTTP response and returned data structure.

### Validation Checks
1. HTTP response status must be 200.
2. The API response must contain a `rates` field.
3. The response must contain historical exchange-rate observations.
4. The returned base currency must be USD.

### Expected Result
A valid API response containing historical USD/INR exchange rates.


In [2]:

# Historical date range
START_DATE = "2026-05-21"
END_DATE = "2026-10-13"

# Currency configuration
BASE_CURRENCY = "USD"
QUOTE_CURRENCY = "INR"

# Build API endpoint
url = f"{FRANKFURTER_BASE_URL}/{START_DATE}..{END_DATE}"

params = {
    "base": BASE_CURRENCY,
    "symbols": QUOTE_CURRENCY
}

# Send API request
response = requests.get(
    url,
    params=params,
    timeout=30
)

print("HTTP Status:", response.status_code)

# Stop if the request failed
response.raise_for_status()

# Convert response to JSON
fx_data = response.json()

# Validate response structure
assert "rates" in fx_data, "Missing rates field"
assert fx_data.get("base") == BASE_CURRENCY, "Unexpected base currency"
assert len(fx_data["rates"]) > 0, "No exchange-rate records returned"

print("Base Currency:", fx_data.get("base"))
print("Start Date:", fx_data.get("start_date"))
print("End Date:", fx_data.get("end_date"))
print("Total Observations:", len(fx_data["rates"]))

# Preview first three observations
print("\nSample Exchange Rates:")

for date, rates in list(sorted(fx_data["rates"].items()))[:3]:
    print(date, rates)


HTTP Status: 200
Base Currency: USD
Start Date: 2026-05-21
End Date: 2026-10-13
Total Observations: 104

Sample Exchange Rates:
2026-05-21 {'INR': 96.21}
2026-05-22 {'INR': 95.7}
2026-05-25 {'INR': 95.24}



## 3. Convert USD/INR API Response into a DataFrame

### Objective
Transform the nested Frankfurter API JSON response into a structured Pandas DataFrame suitable for validation, SQL storage, and financial analysis.

### Method
- Extract the observation date and INR exchange rate.
- Add base currency, quote currency, and source information.
- Convert dates and exchange rates into appropriate data types.
- Sort records chronologically.

### Validation
1. Confirm that the DataFrame contains 104 observations.
2. Check that the required columns exist.
3. Inspect data types and sample records.

### Expected Result
A structured USD/INR dataset with one row per exchange-rate observation.


In [3]:

# Convert nested API response into structured records
fx_records = []

for observation_date, currencies in fx_data["rates"].items():
    fx_records.append({
        "observation_date": observation_date,
        "base_currency": BASE_CURRENCY,
        "quote_currency": QUOTE_CURRENCY,
        "exchange_rate": currencies[QUOTE_CURRENCY],
        "source": "Frankfurter"
    })

# Create DataFrame
df_usd_inr = pd.DataFrame(fx_records)

# Apply appropriate data types
df_usd_inr["observation_date"] = pd.to_datetime(
    df_usd_inr["observation_date"]
)

df_usd_inr["exchange_rate"] = pd.to_numeric(
    df_usd_inr["exchange_rate"]
)

# Sort by date
df_usd_inr = (
    df_usd_inr
    .sort_values("observation_date")
    .reset_index(drop=True)
)

# Display results
print("Total Records:", len(df_usd_inr))
print("\nColumn Names:")
print(df_usd_inr.columns.tolist())

print("\nData Types:")
print(df_usd_inr.dtypes)

print("\nFirst Five Records:")
display(df_usd_inr.head())


Total Records: 104

Column Names:
['observation_date', 'base_currency', 'quote_currency', 'exchange_rate', 'source']

Data Types:
observation_date    datetime64[us]
base_currency                  str
quote_currency                 str
exchange_rate              float64
source                         str
dtype: object

First Five Records:


,observation_date,base_currency,quote_currency,exchange_rate,source
0,2026-05-21,USD,INR,96.21,Frankfurter
1,2026-05-22,USD,INR,95.70,Frankfurter
2,2026-05-25,USD,INR,95.24,Frankfurter
3,2026-05-26,USD,INR,95.69,Frankfurter
4,2026-05-27,USD,INR,95.69,Frankfurter



## 4. Reusable FX API Extraction Function

### Business Objective
Collect historical exchange rates for all three currencies used in the company's international import payments: USD, EUR, and GBP.

### Technical Objective
Create a reusable Python function that extracts and standardizes historical exchange-rate data from the Frankfurter API.

### Method
1. Accept the base currency and historical date range.
2. Request exchange rates against INR.
3. Validate the HTTP response and expected JSON structure.
4. Convert the API response into a structured DataFrame.
5. Return standardized records for further processing.

### Expected Result
A reusable extraction function that produces consistent data structures for USD/INR, EUR/INR, and GBP/INR.


In [4]:

def extract_fx_rates(base_currency, start_date, end_date):
    """
    Extract historical exchange rates against INR
    from the Frankfurter API.
    """

    url = f"{FRANKFURTER_BASE_URL}/{start_date}..{end_date}"

    params = {
        "base": base_currency,
        "symbols": "INR"
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    # Validate API response
    if data.get("base") != base_currency:
        raise ValueError(
            f"Unexpected base currency: {data.get('base')}"
        )

    if not data.get("rates"):
        raise ValueError(
            f"No exchange-rate data returned for {base_currency}"
        )

    # Convert JSON into standardized records
    records = []

    for date, rates in data["rates"].items():
        if "INR" not in rates:
            raise ValueError(
                f"Missing INR rate for {base_currency} on {date}"
            )

        records.append({
            "observation_date": date,
            "base_currency": base_currency,
            "quote_currency": "INR",
            "exchange_rate": rates["INR"],
            "source": "Frankfurter"
        })

    df = pd.DataFrame(records)

    df["observation_date"] = pd.to_datetime(
        df["observation_date"]
    )

    df["exchange_rate"] = pd.to_numeric(
        df["exchange_rate"]
    )

    df = (
        df.sort_values("observation_date")
        .reset_index(drop=True)
    )

    return df


print("Reusable FX extraction function created successfully.")


Reusable FX extraction function created successfully.



## 5. Extract and Consolidate Historical FX Data

### Business Objective
Build a consolidated foreign-exchange dataset covering the three currencies used for international import payments.

### Method
1. Extract historical USD/INR, EUR/INR, and GBP/INR rates using the reusable API function.
2. Combine the individual currency datasets into one DataFrame.
3. Sort observations by currency and date.
4. Compare record counts across currency pairs.

### Validation
- All three currency pairs must be present.
- Each currency pair must contain historical observations.
- The combined record count must equal the sum of the individual counts.
- Each record must identify its base and quote currencies.

### Expected Result
One consolidated historical FX dataset ready for data-quality validation and PostgreSQL ingestion.


In [5]:

# Define currencies required by the Treasury team
CURRENCIES = ["USD", "EUR", "GBP"]

# Store individual currency DataFrames
fx_dataframes = []

for currency in CURRENCIES:

    df_currency = extract_fx_rates(
        base_currency=currency,
        start_date=START_DATE,
        end_date=END_DATE
    )

    fx_dataframes.append(df_currency)

    print(
        f"{currency}/INR | Records: {len(df_currency)}"
    )

# Consolidate all currency datasets
df_fx = pd.concat(
    fx_dataframes,
    ignore_index=True
)

# Sort by currency and observation date
df_fx = (
    df_fx
    .sort_values(["base_currency", "observation_date"])
    .reset_index(drop=True)
)

# Initial structural checks
expected_pairs = set(CURRENCIES)
actual_pairs = set(df_fx["base_currency"].unique())

assert actual_pairs == expected_pairs, "Currency pair mismatch"
assert df_fx["quote_currency"].eq("INR").all(), "Unexpected quote currency"
assert len(df_fx) == sum(len(df) for df in fx_dataframes), "Record count mismatch"

print("\nConsolidated FX Records:", len(df_fx))

print("\nRecords by Currency:")
print(df_fx.groupby("base_currency").size())

print("\nFirst Five Records:")
display(df_fx.head())


USD/INR | Records: 104
EUR/INR | Records: 104
GBP/INR | Records: 104

Consolidated FX Records: 312

Records by Currency:
base_currency
EUR    104
GBP    104
USD    104
dtype: int64

First Five Records:


,observation_date,base_currency,quote_currency,exchange_rate,source
0,2026-05-21,EUR,INR,111.5885,Frankfurter
1,2026-05-22,EUR,INR,110.9585,Frankfurter
2,2026-05-25,EUR,INR,110.8870,Frankfurter
3,2026-05-26,EUR,INR,111.3200,Frankfurter
4,2026-05-27,EUR,INR,111.3650,Frankfurter



## 6. Save Raw FX Data and Ingestion Metadata

### Objective
Preserve the extracted foreign-exchange data and record ingestion details for auditing, reproducibility, and future validation.

### Method
- Save the consolidated FX dataset as a CSV file.
- Save ingestion metadata as a JSON file.
- Record the data source, currency pairs, extraction period, retrieval timestamp, and record counts.

### Validation
1. Confirm both files are created successfully.
2. Verify the saved CSV record count matches the in-memory DataFrame.
3. Confirm the saved CSV contains the expected columns.

### Expected Result
Two files in the project's `Data/raw/` directory:
- `fx_rates_raw.csv`
- `fx_ingestion_metadata.json`

### Note
The CSV is an uncleaned, standardized extract of the API observations. It is not the original JSON response. Detailed data-quality checks will be performed separately.


In [6]:

import json
from datetime import datetime, timezone

# Define raw data directory
RAW_DATA_DIR = PROJECT_ROOT / "Data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Define output paths
csv_path = RAW_DATA_DIR / "fx_rates_raw.csv"
metadata_path = RAW_DATA_DIR / "fx_ingestion_metadata.json"

# Save standardized, uncleaned API observations
df_fx.to_csv(csv_path, index=False)

# Build ingestion metadata
metadata = {
    "dataset": "historical_fx_rates",
    "source": "Frankfurter API",
    "api_base_url": FRANKFURTER_BASE_URL,
    "currency_pairs": ["USD/INR", "EUR/INR", "GBP/INR"],
    "start_date": START_DATE,
    "end_date": END_DATE,
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "total_records": len(df_fx),
    "records_by_currency": {
        currency: int(count)
        for currency, count in df_fx.groupby("base_currency").size().items()
    },
    "columns": df_fx.columns.tolist()
}

# Save metadata
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

# Verify the saved CSV
df_saved = pd.read_csv(csv_path)

assert len(df_saved) == len(df_fx), "Saved CSV record count mismatch"
assert df_saved.columns.tolist() == df_fx.columns.tolist(), "Column mismatch"

print("Raw FX Data: SAVED")
print("CSV Path:", csv_path)
print("Metadata Path:", metadata_path)
print("Saved Records:", len(df_saved))
print("CSV Validation: PASSED")


Raw FX Data: SAVED
CSV Path: E:\BA Projects\Financial_Market_Intelligence_Analytics\Data\raw\fx_rates_raw.csv
Metadata Path: E:\BA Projects\Financial_Market_Intelligence_Analytics\Data\raw\fx_ingestion_metadata.json
Saved Records: 312
CSV Validation: PASSED


In [2]:

import pandas as pd

raw_fx_path = project_root / "Data" / "raw" / "fx_rates_raw.csv"
metadata_path = project_root / "Data" / "raw" / "fx_ingestion_metadata.json"

raw_fx_df = pd.read_csv(raw_fx_path)

logger.info(
    "Raw FX dataset verified | Records: %s | File: %s",
    len(raw_fx_df),
    raw_fx_path.name
)

logger.info(
    "FX ingestion metadata exists: %s",
    metadata_path.is_file()
)

logger.info("Raw FX ingestion output verification completed")


2026-11-02 16:01:36,071 | INFO | FX_API_Ingestion | Raw FX dataset verified | Records: 312 | File: fx_rates_raw.csv
2026-11-02 16:01:36,076 | INFO | FX_API_Ingestion | FX ingestion metadata exists: True
2026-11-02 16:01:36,078 | INFO | FX_API_Ingestion | Raw FX ingestion output verification completed
